# 11 · Ampliación opcional. Diagnóstico de errores

En el cuaderno 11 calculamos un error medio. Aquí repetimos ese modelo y miramos **en qué filas se equivoca más**. Se puede ejecutar por separado.

## 0 · Preparar el entorno

Elige un kernel de **Python 3.11 o 3.12** con **Java 17**. En Jupyter o VS Code usa el mismo entorno que en las prácticas anteriores. En Colab, comprueba la versión con `!python --version`. Si muestra Python 3.13, en **Entorno de ejecución → Cambiar tipo de entorno de ejecución** selecciona la versión **2026.07**, que usa Python 3.12. Si no tienes Java 17, ejecuta en una celda de Colab `!apt-get -qq update && apt-get -qq install -y openjdk-17-jre-headless` y, en otra celda, `import os; os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-17-openjdk-amd64"`.

Si faltan las bibliotecas, ejecuta en una celda `%pip install "pyspark>=3.5,<3.6" "numpy<2"` y reinicia el kernel si te lo pide. Puedes consultar la [guía del curso](../01_environment_pyspark.md). Ejecuta las celdas de arriba abajo. Los datos de ejemplo están dentro del notebook y el cálculo se hace en el equipo donde corre el kernel.

In [ ]:
from pyspark.sql import SparkSession, functions as F
from pyspark.ml.feature import VectorAssembler
from pyspark.ml.regression import LinearRegression

spark = (SparkSession.builder.master('local[2]').appName('Diagnostico de regresion')
         .config('spark.sql.shuffle.partitions', '4').getOrCreate())
print(f'Spark {spark.version}')

## 1 · Crear y revisar la tabla de ejemplo

Una fila representa a una persona ficticia. Las horas de formación previa y la jornada semanal se conocen antes de intentar predecir un resultado. El salario mensual y la finalización del curso son dos resultados diferentes. Usamos una semilla para generar la misma tabla en los cuadernos 11, 12 y 14.

La encuesta es **simulada**. Sirve para practicar el método y sus métricas.

In [ ]:
from random import Random

azar = Random(42)
filas = []

for persona in range(1, 361):
    # Tres perfiles de partida hacen visible la idea de segmentación.
    experiencia_base, jornada_base, formacion_base = azar.choice([
        (3, 24, 12), (8, 35, 28), (18, 40, 8),
    ])
    experiencia = max(0, experiencia_base + azar.randint(-3, 3))
    jornada = jornada_base + azar.randint(-4, 4)
    formacion = max(0, formacion_base + azar.randint(-8, 8))
    salario = round(650 + 60 * experiencia + 23 * jornada + azar.gauss(0, 220), 2)
    # La formación previa ayuda y una jornada más larga deja menos tiempo.
    puntuacion = formacion - 0.45 * jornada + azar.gauss(0, 9)
    completa = int(puntuacion >= 1)
    filas.append((persona, experiencia, jornada, formacion, salario, completa))

datos = spark.createDataFrame(filas, [
    'persona', 'anios_experiencia', 'horas_jornada', 'horas_formacion',
    'salario_mensual', 'completa_curso',
])

print(f'Filas: {datos.count()}')
datos.show(5)

## 2 · Elegir lo que queremos predecir

La variable objetivo es `salario_mensual`. La experiencia y las horas semanales de jornada son las dos entradas. `VectorAssembler` coloca las entradas de cada fila en la columna `variables`, en ese mismo orden. Spark espera un vector para entrenar el modelo.

In [ ]:
entradas = ['anios_experiencia', 'horas_jornada']
ensamblador = VectorAssembler(inputCols=entradas, outputCol='variables')
preparados = ensamblador.transform(datos)
preparados.select('salario_mensual', 'variables').show(5)

## 3 · Separar entrenamiento y prueba

El modelo aprende solo con **entrenamiento**. Reservamos **prueba** para simular lo que ocurre cuando llegan filas nuevas. El reparto es aproximadamente 80/20. La semilla permite repetir la división con los mismos datos.

In [ ]:
entrenamiento, prueba = preparados.randomSplit([0.8, 0.2], seed=42)
print(f'Entrenamiento: {entrenamiento.count()} · Prueba: {prueba.count()}')

## 4 · Ajustar el modelo

Entrenamos la misma regresión lineal del cuaderno 11 y obtenemos predicciones en la parte reservada.

In [ ]:
modelo = LinearRegression(
    featuresCol='variables', labelCol='salario_mensual', predictionCol='prediccion',
).fit(entrenamiento)
predicciones = modelo.transform(prueba)
predicciones.select('anios_experiencia', 'horas_jornada', 'salario_mensual',
                    F.round('prediccion', 2).alias('salario_estimado')).show(8)

## 5 · Leer el error de cada fila

Restamos la predicción al salario observado. Un residuo positivo indica que el modelo se quedó corto. El error absoluto mide cuánto se desvió en euros. Ordenamos por ese error para revisar los casos más difíciles.

In [ ]:
errores = (predicciones.withColumn('residuo', F.col('salario_mensual') - F.col('prediccion'))
           .withColumn('error_absoluto', F.abs('residuo')))
errores.orderBy(F.desc('error_absoluto')).select(
    'persona', 'anios_experiencia', 'horas_jornada', 'salario_mensual',
    F.round('prediccion', 2).alias('estimado'),
    F.round('residuo', 2).alias('residuo'),
).show(8)

## 6 · Comprobar si el error cambia por jornada

Separamos las jornadas de menos de 30 horas semanales de las demás. Leemos el número de filas junto al error medio para dar contexto a la comparación.

In [ ]:
(errores.withColumn('jornada', F.when(F.col('horas_jornada') < 30, 'Menos de 30 horas')
                     .otherwise('30 horas o más'))
 .groupBy('jornada')
 .agg(F.count('*').alias('personas'),
      F.round(F.avg('residuo'), 2).alias('residuo_medio'),
      F.round(F.avg('error_absoluto'), 2).alias('error_absoluto_medio'))
 .show(truncate=False))

**Interpretación.** Un residuo medio positivo sugiere que el modelo suele quedarse corto en ese grupo de prueba. En un proyecto real, revisaríamos más datos antes de atribuir el patrón a la jornada.

## Cerrar la sesión

Liberamos los recursos de Spark al terminar este cuaderno.

In [ ]:
spark.stop()

## Navegación

[Volver al README](../README.md) · [Regresión lineal guiada](11_census_linear_regression.ipynb) · [Siguiente: Clasificación con regresión logística](12_census_logistic_regression.ipynb)

Documentación de Spark ML. [Regresión y clasificación](https://spark.apache.org/docs/3.5.6/ml-classification-regression.html) · [Preparación de variables](https://spark.apache.org/docs/3.5.6/ml-features.html)